<a href="https://colab.research.google.com/github/him2079/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/him2079/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
%cd /content
!git clone https://github.com/him2079/flyrank-ml-internship.git
%cd flyrank-ml-internship

!pip install -q duckdb

import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()

con.install_extension("httpfs")
con.load_extension("httpfs")

con.execute(f"""
    CREATE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{hf_token}'
    );
""")

/content
fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.
/content/flyrank-ml-internship


In [12]:
con.execute("""
    DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

One row = one content item, on one day, for one client (grain: client_hash_id + content_hash_id + report_date, from fact_content_daily_performance). I'm working over month=2026-03 as my development window — a mid-panel month, not the final month (_sample), since that's a sealed test month and using it now would mean peeking at the future I'm supposed to predict.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

Features: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_engaged_sessions — all observed signals knowable before the decision point.
Label/proxy: whether clicks declined from the prior half of March to the later half (clicks_late < clicks_early) — a future-outcome label built from real data, not a same-window snapshot.
Context: client_hash_id, content_hash_id — used for joins and grouping only, never as features, since the codes themselves carry no meaning.
Excluded: any FlyRank product decision fields like health_score, priority_score, or action_type. These aren't shipped in the warehouse data, but I'm naming the exclusion explicitly — if I ever rebuilt one, feeding it back in as a feature would let the model copy an existing answer instead of finding real signal.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)



Confirms one row really is one client × content item × day, as claimed in Section 1.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q1 = con.execute("""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as row_count
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()
q1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,row_count


Zero duplicate rows returned for client_hash_id + content_hash_id + report_date — confirming the grain claim holds.

Checking the actual size and date span of this slice.

In [16]:
q2 = con.execute("""
    SELECT COUNT(*) as total_rows, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
q2

,total_rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


The March 2026 slice contains 9,841,378 rows, spanning the full month from 2026-03-01 to 2026-03-31 — matching the expected date window exactly.

Checking GA4 availability specifically, since not every client has GA4 tracking active for their full history.

In [17]:
q3 = con.execute("""
    SELECT COUNT(*) as available_rows
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE ga4_data_available IS TRUE
""").df()
q3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,413966


Of the 9,841,378 total rows, only 413,966 (about 4.2%) have GA4 data available. GSC coverage is far denser than GA4 coverage in this slice — any feature relying on sessions or engagement will only be reliable for this smaller subset.

Building five features from the prior half of March (1–15) and a future-outcome label from the later half (16–31), so the label reflects what actually happened next, not a same-window snapshot.

In [18]:
features_df = con.execute("""
WITH early AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) as impressions_early,
        SUM(gsc_clicks) as clicks_early,
        AVG(gsc_avg_position) as avg_position_early,
        SUM(ga4_sessions) as sessions_early,
        SUM(ga4_engaged_sessions) as engaged_sessions_early
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
    GROUP BY 1,2
),
late AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_clicks) as clicks_late
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'
    GROUP BY 1,2
)
SELECT e.*, l.clicks_late,
    CASE WHEN l.clicks_late < e.clicks_early THEN 1 ELSE 0 END as is_declining
FROM early e
JOIN late l ON e.client_hash_id = l.client_hash_id AND e.content_hash_id = l.content_hash_id
WHERE e.clicks_early > 0
""").df()

features_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_early,clicks_early,avg_position_early,sessions_early,engaged_sessions_early,clicks_late,is_declining
0,client_3ffa76342f366962,content_a3a68ca6161f8c25,1.0,1.0,4.000000,0.0,0.0,0.0,1
1,client_3ffa76342f366962,content_7822536c538cfd25,18.0,1.0,6.012500,1.0,0.0,0.0,1
2,client_3ffa76342f366962,content_fd41b905fa6f913a,30.0,2.0,2.230769,0.0,0.0,1.0,1
3,client_3ffa76342f366962,content_6337d6ec49266f69,2.0,1.0,5.000000,0.0,0.0,0.0,1
4,client_3ffa76342f366962,content_fa91a12aee99af90,11.0,1.0,4.611111,0.0,0.0,0.0,1


Five features, each knowable at the decision point (end of March 15):

1.impressions_early — GSC impressions summed only over March 1–15, before the decision point.
2.clicks_early — GSC clicks summed over the same prior window.
3.avg_position_early — average search position across the prior window.
4.sessions_early — GA4 sessions in the prior window (available only for the GA4-covered subset).
5.engaged_sessions_early — engaged sessions in the prior window, same GA4 availability caveat.

The label, is_declining, compares clicks_late (March 16–31) against clicks_early — a future outcome relative to the feature window, not a same-window snapshot.

The trap: deliberately including clicks_late — the same window the label is derived from — as a feature, to see the score inflate.

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_leaky = features_df[['clicks_late']].fillna(0)
y = features_df['is_declining']
X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
model_leaky = LogisticRegression().fit(X_train, y_train)
print("Leaky score:", model_leaky.score(X_test, y_test))

Leaky score: 0.6383163167019842


In [20]:
X_honest = features_df[['impressions_early','clicks_early','avg_position_early','sessions_early','engaged_sessions_early']].fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.2, random_state=42)
model_honest = LogisticRegression().fit(X_train, y_train)
print("Honest score:", model_honest.score(X_test, y_test))

Honest score: 0.5602003467539973


Including clicks_late as a feature gave a score of 0.638. Removing it and using only prior-window features gave 0.560. The gap is real but modest, not dramatic — worth noting honestly rather than overstating it. Even a small amount of leaked future information inflates performance, and the honest 0.560 score, not 0.638, is the number that reflects what this data can actually predict without cheating.

## 4. Data limits

This slice has an unbalanced panel — clients have different amounts of tracking history, and only 9 of 70 clients have 12+ months, limiting seasonality analysis. GA4 coverage is thin: only 413,966 of 9,841,378 rows (about 4.2%) have GA4 data available, so session- and engagement-based features are only reliable for a small subset of this slice — rows before a client's GA4 start date are search-data-only. AI-session data is extremely sparse across the whole warehouse (30,177 rows out of 78.8 million per the lane guide), so any AI-referral signal should be treated as directional at best. Finally, even a modest amount of leaked future information (the clicks_late trap) inflated the score from 0.567 to 0.648 — a reminder that this slice's honest predictive power is limited, and any stronger result later needs to be checked carefully for the same kind of leak.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.